In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

def decode_fen(fen: str) -> torch.Tensor:
    parts = fen.split(" ")
    side = parts[1]
    castling = parts[2]
    en_passant = parts[3]
    fifty_move = int(parts[4]) if len(parts) > 4 else 0
    board_str = parts[0]

    board = [0.0] * 768

    rank = 7
    file = 0
    for c in board_str:
        if c == "/":
            rank -= 1
            file = 0
        elif c.isdigit():
            file += int(c)
        else:
            color = 0 if c.isupper() else 1
            piece = "pbnrqk".index(c.lower())
            square = rank * 8 + file
            board[color * 384 + piece * 64 + square] = 1.0
            file += 1

    side_node = [1.0 if side == "w" else 0.0]

    castling_nodes = [
        1.0 if "K" in castling else 0.0,
        1.0 if "Q" in castling else 0.0,
        1.0 if "k" in castling else 0.0,
        1.0 if "q" in castling else 0.0,
    ]

    ep_nodes = [0.0] * 8
    if en_passant != "-":
        ep_nodes["abcdefgh".index(en_passant[0])] = 1.0
    fifty_node = [min(fifty_move, 100) / 100.0]

    inputs = board + side_node + castling_nodes + ep_nodes + fifty_node

    return torch.tensor(inputs, dtype=torch.float32)


class ChessNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(782, 256),  # 6 * 2 * 64 + 1 + 4 + 8 + 1
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net()

class ChessDataset(Dataset):
    def __init__(self, filepath):
        self.samples = []
        with open(filepath, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                fen, score = line.rsplit(" | ", 1)
                self.samples.append((fen, float(score)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fen, score = self.samples[idx]
        x = decode_fen(fen)
        y = torch.tensor([score], dtype=torch.float32)
        return x, y

In [ ]:
# Actual training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{device}")

dataset = ChessDataset("/home/darko/projects/chess-monkfish/positions.txt")
train_size = int(0.95 * len(dataset))  # train on 95% of data
val_size = len(dataset) - train_size  # test on remainig 5% for validation
train_set, val_set = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=2048, shuffle=True, pin_memory=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=2048,  pin_memory=True, num_workers=0)

model = ChessNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()


for epoch in range(20):
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model.net(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model.net(x)
            val_loss += criterion(pred, y).item()

        print(f"Epoch {epoch+1:02d} | "
          f"Train: {train_loss/len(train_loader):.4f} | "
          f"Val: {val_loss/len(val_loader):.4f}")

torch.save(model.state_dict(), "chess_net.pt")



cuda
Epoch 01 | Train: 0.6572 | Val: 0.6466
Epoch 02 | Train: 0.6426 | Val: 0.6440
Epoch 03 | Train: 0.6383 | Val: 0.6439
Epoch 04 | Train: 0.6353 | Val: 0.6428
Epoch 05 | Train: 0.6327 | Val: 0.6430
Epoch 06 | Train: 0.6306 | Val: 0.6432
Epoch 07 | Train: 0.6288 | Val: 0.6439
Epoch 08 | Train: 0.6271 | Val: 0.6446
Epoch 09 | Train: 0.6258 | Val: 0.6448
Epoch 10 | Train: 0.6244 | Val: 0.6467
Epoch 11 | Train: 0.6232 | Val: 0.6464
Epoch 12 | Train: 0.6222 | Val: 0.6476
Epoch 13 | Train: 0.6212 | Val: 0.6483
Epoch 14 | Train: 0.6203 | Val: 0.6483
Epoch 15 | Train: 0.6194 | Val: 0.6498
Epoch 16 | Train: 0.6185 | Val: 0.6507
Epoch 17 | Train: 0.6178 | Val: 0.6504
Epoch 18 | Train: 0.6171 | Val: 0.6508
Epoch 19 | Train: 0.6163 | Val: 0.6518
Epoch 20 | Train: 0.6156 | Val: 0.6528


ModuleNotFoundError: No module named 'matplotlib'